# Question 2 — MLflow Experiment Comparison (MNIST + MLP)
### AI Operations (AIOps) — MLflow Deep Dive

**Objective:** train an `MLPClassifier` on MNIST, run six experiments varying `learning_rate_init`
and `batch_size`, log per-epoch `train_loss` and `val_accuracy` to MLflow, and compare all six runs.

> **Prerequisite:** a local MLflow Tracking Server must already be running:
> ```bash
> mlflow server --backend-store-uri sqlite:///mlflow.db \
>     --default-artifact-root ./mlruns --host 0.0.0.0 --port 5000 --allowed-hosts "*" --cors-allowed-origins "http://localhost:5000, http://127.0.0.1:5000"
> ```
> Run that command in a separate terminal *before* executing the cells below, then leave it running.

## Step 0 — Setup

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

2026/08/29 18:47:08 INFO mlflow.tracking.fluent: Experiment with name 'mnist-mlp-classifier' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000


## Step 1 — Load MNIST and prepare train/validation splits

In [2]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
X = X / 255.0
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=6000, test_size=1000, random_state=42, stratify=y
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)

X_train: (6000, 784)  X_test: (1000, 784)


## Step 2 — Instrument training: manual per-epoch logging

In [4]:
def train_and_log(learning_rate_init=0.001, batch_size=64, hidden_layer_sizes=(100,),
                   n_epochs=30, run_name=None):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("n_epochs", n_epochs)
        mlflow.log_param("model_type", "MLPClassifier")

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            learning_rate_init=learning_rate_init,
            batch_size=batch_size,
            warm_start=True,
            max_iter=1,
            random_state=42,
        )

        for epoch in range(n_epochs):
            model.fit(X_train, y_train)
            train_loss = model.loss_
            train_acc = accuracy_score(y_train, model.predict(X_train))
            val_acc = accuracy_score(y_test, model.predict(X_test))

            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_accuracy", train_acc, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)

        mlflow.log_metric("final_val_accuracy", val_acc)
        
        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id}  |  lr={learning_rate_init}  batch_size={batch_size}  final_val_acc={val_acc:.4f}")
        return run_id

baseline_run_id = train_and_log(0.001, 64, run_name="mlp-lr0.001-bs64")

/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't conv

Logged run 6c3f0e491b4146eaab76d72900328b2f  |  lr=0.001  batch_size=64  final_val_acc=0.9360
🏃 View run mlp-lr0.001-bs64 at: http://localhost:5000/#/experiments/1/runs/6c3f0e491b4146eaab76d72900328b2f
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Step 3 — Sweep: six runs varying `learning_rate_init` and `batch_size`

In [5]:
sweep_run_ids = []
for lr in [0.001, 0.01, 0.1]:
    for bs in [64, 128, 256]:
        rid = train_and_log(learning_rate_init=lr, batch_size=bs,
                             run_name=f"mlp-lr{lr}-bs{bs}")
        sweep_run_ids.append(rid)

print("Sweep run IDs:", sweep_run_ids)

/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't conv

Logged run 5f54824e005544e4803a57e02665c5d7  |  lr=0.001  batch_size=64  final_val_acc=0.9360
🏃 View run mlp-lr0.001-bs64 at: http://localhost:5000/#/experiments/1/runs/5f54824e005544e4803a57e02665c5d7
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't conv

Logged run f226ac22135f4339a64a89af56f69ec8  |  lr=0.001  batch_size=128  final_val_acc=0.9370
🏃 View run mlp-lr0.001-bs128 at: http://localhost:5000/#/experiments/1/runs/f226ac22135f4339a64a89af56f69ec8
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't conv

...

Logged run efd586c78ffb43f480c1a24c10ea3aca  |  lr=0.1  batch_size=64  final_val_acc=0.7570
🏃 View run mlp-lr0.1-bs64 at: http://localhost:5000/#/experiments/1/runs/efd586c78ffb43f480c1a24c10ea3aca
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't conv

Logged run 69ecf6eb4d0545e58465382e1362332a  |  lr=0.1  batch_size=128  final_val_acc=0.7820
🏃 View run mlp-lr0.1-bs128 at: http://localhost:5000/#/experiments/1/runs/69ecf6eb4d0545e58465382e1362332a
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/mudda-manikanta-pruthvi-raj/.local/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't conv

Logged run 4a300f0ede6644bd88c6351c0e7ec763  |  lr=0.1  batch_size=256  final_val_acc=0.7660
🏃 View run mlp-lr0.1-bs256 at: http://localhost:5000/#/experiments/1/runs/4a300f0ede6644bd88c6351c0e7ec763
🧪 View experiment at: http://localhost:5000/#/experiments/1
Sweep run IDs: ['5f54824e005544e4803a57e02665c5d7', 'f226ac22135f4339a64a89af56f69ec8', '1fa603be14974488b2625553275038a5', '99dbd5a09fe349218e8aa3dfb87859ca', '806f21f399b04c2e9b6234f753c88735', '96d8726863d74954b9cf2472dab62778', 'efd586c78ffb43f480c1a24c10ea3aca', '69ecf6eb4d0545e58465382e1362332a', '4a300f0ede6644bd88c6351c0e7ec763']


## Step 4 — Find the best run with `mlflow.search_runs()`

In [6]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp-classifier"],
    order_by=["metrics.final_val_accuracy DESC"],
)

display_cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.learning_rate_init", "params.batch_size",
    "metrics.final_val_accuracy", "metrics.train_loss"
)]
print(runs_df[display_cols].head(10).to_string(index=False))

best_run = runs_df.iloc[0]
print(f"\nBest run: {best_run['run_id']}  (val_accuracy={best_run['metrics.final_val_accuracy']:.4f})")

                          run_id  metrics.train_loss  metrics.final_val_accuracy params.batch_size params.learning_rate_init tags.mlflow.runName
96d8726863d74954b9cf2472dab62778            0.042606                       0.945               256                      0.01    mlp-lr0.01-bs256
806f21f399b04c2e9b6234f753c88735            0.046147                       0.945               128                      0.01    mlp-lr0.01-bs128
f226ac22135f4339a64a89af56f69ec8            0.020870                       0.937               128                     0.001   mlp-lr0.001-bs128
99dbd5a09fe349218e8aa3dfb87859ca            0.066585                       0.936                64                      0.01     mlp-lr0.01-bs64
1fa603be14974488b2625553275038a5            0.059124                       0.936               256                     0.001   mlp-lr0.001-bs256
5f54824e005544e4803a57e02665c5d7            0.008076                       0.936                64                     0.001    ml

## Step 5 — Open the MLflow UI
1. Go to **http://localhost:5000** in your browser.
2. Open the **mnist-mlp-classifier** experiment.
3. Select all 6 runs from this notebook (checkboxes on the left) and click **Compare**.
4. Confirm the run the UI ranks highest matches the `best_run` printed above.

---
### ✅ Deliverable checklist
- [ ] Screenshot of the 6-run comparison view in the MLflow UI
- [ ] The `run_id` of the best run (printed above by `mlflow.search_runs()`)
- [ ] Written analysis (best run, overfitting evidence, which hyperparameter mattered more)